# 08 - Evaluation: Retrieval, Generation, and Judge

## Definition
RAG evaluation measures retrieval quality and generation quality as separate but linked objectives.

## Theory
Poor retrieval creates an upper bound on generation quality regardless of model fluency.

## Motivation
Without robust metrics, improvements may be cosmetic rather than factual.

## Architecture
Query set -> retrieval metrics -> generation metrics -> LLM judge -> hallucination analysis.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [1]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [2]:
bundle = runner.ingest_dataset()
chunking = runner.run_chunking(bundle['documents'][:320], strategy=ChunkingStrategy.RECURSIVE)
runner.index_chunks(chunking.chunks, reset=True)

queries = bundle['queries'][:24]
summary, frames = runner.evaluator.run_full_evaluation(
    queries=queries,
    top_k=6,
    retrieval_limit=18,
    generation_limit=6,
    judge_limit=6,
)

{
    'retrieval': summary.retrieval,
    'generation': summary.generation,
    'judge': summary.judge,
}


[nltk_data] Downloading package wordnet to /home/ahmad/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ahmad/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ahmad/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'retrieval': RetrievalMetricSummary(precision_at_k=0.0, recall_at_k=0.0, f1_at_k=0.0, mrr=0.0, ndcg=0.0, num_queries=18),
 'generation': GenerationMetricSummary(exact_match=0.0, bleu=0.0, rouge1=0.014249684741488022, rougeL=0.014249684741488022, meteor=0.027952480782669466, bertscore_f1=0.5873654007911682, num_examples=5),
 'judge': JudgeMetricSummary(relevance=1.0, correctness=1.0, groundedness=1.0, completeness=1.0, faithfulness=1.0, num_examples=6)}

In [3]:
hall = runner.evaluator.evaluate_hallucination_reduction(queries=queries[:8], max_queries=8)
hall[['groundedness_delta', 'rag_faithfulness', 'no_rag_faithfulness']].describe()


,groundedness_delta,rag_faithfulness,no_rag_faithfulness
count,8.000000,8.00000,8.000000
mean,-2.750000,1.37500,4.875000
std,1.164965,1.06066,0.353553
min,-4.000000,1.00000,4.000000
25%,-3.000000,1.00000,5.000000
50%,-3.000000,1.00000,5.000000
75%,-3.000000,1.00000,5.000000
max,0.000000,4.00000,5.000000
